# 04. Exploratory Data Analysis & Diagnostic Intelligence
Connecting to the SQLite analytical data warehouse and generating diagnostic charts.


In [ ]:
import sys
sys.path.append("..")
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from src.database import LogisticsDatabase

db = LogisticsDatabase()


### Weekly OTD Performance & Volume Trajectory


In [ ]:
sql_wk = """
SELECT year_week, COUNT(*) AS volume, AVG(on_time_flag)*100.0 AS otd_pct
FROM fact_shipments WHERE shipment_status != 'Lost / Exception'
GROUP BY year_week ORDER BY year_week;
"""
df_wk = db.run_query(sql_wk)
df_wk.head(10)


### Hub-Level Deterioration Analysis


In [ ]:
sql_hubs = """
SELECT 
    origin_hub_id,
    AVG(CASE WHEN is_decline_period = 0 THEN on_time_flag END)*100.0 AS baseline_otd,
    AVG(CASE WHEN is_decline_period = 1 THEN on_time_flag END)*100.0 AS decline_otd
FROM fact_shipments WHERE shipment_status != 'Lost / Exception'
GROUP BY origin_hub_id;
"""
df_hubs = db.run_query(sql_hubs)
df_hubs["decline_pp"] = df_hubs["decline_otd"] - df_hubs["baseline_otd"]
df_hubs.sort_values(by="decline_pp")
